# SDXL LoRA finetuning on Kaggle

Enable a GPU and Internet in Kaggle settings, and attach your image dataset.
Run Steps 0–1 once, **restart the kernel**, then run from Step 2 onward.
Setup uses sd-scripts **v0.10.6** and its own requirements, with no fallback to main.
This is an environment update; rerun all comparison arms with the same environment.

The original experiment defaults are retained: 512×512, rank 16, alpha 8,
learning rate 1e-4, 500 optimizer steps, batch 1 × accumulation 4 on one GPU,
50 warmup steps, seed 42, and the uniform caption `ohwx person`.
A referenced “Table 5” was not supplied, so exact agreement with it is unverified.
SDXL normally benefits from higher resolution; keep 512 for this comparison unless
deliberately changing the protocol for every run. Resizing/cropping can affect cloak
perturbations, so keep image preprocessing identical across clean and cloaked runs.

In [ ]:
from pathlib import Path
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Enable a Kaggle GPU accelerator before continuing.")
subprocess.run([
    "nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"
], check=True)
input_root = Path("/kaggle/input")
if not input_root.is_dir():
    raise RuntimeError("This notebook expects Kaggle paths under /kaggle/input.")
print("Attached inputs:", [p.name for p in input_root.iterdir()])

## Step 1 — Install the trainer once

Install from the trainer directory so its editable `-e .` requirement resolves correctly.
Preserve Kaggle's installed torch/torchvision/torchaudio versions using pip constraints;
do not install xformers because this notebook uses PyTorch SDPA.
PEFT is pinned for compatibility if it is already present in Kaggle's environment.
NumPy 1.26 avoids the NumPy 2 ABI mismatch with this release's OpenCV wheel.
Any clone/install failure stops the cell. Restart the kernel after installation.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("This dependency set targets Python 3.10–3.12.")
trainer_dir = Path("/kaggle/working/sd-scripts-v0.10.6")
trainer_ref = "v0.10.6"
if not trainer_dir.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", trainer_ref,
        "https://github.com/kohya-ss/sd-scripts.git", str(trainer_dir)
    ], check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=trainer_dir, text=True).strip()
tagged = subprocess.check_output([
    "git", "rev-parse", f"{trainer_ref}^{{commit}}"
], cwd=trainer_dir, text=True).strip()
if head != tagged or subprocess.check_output([
    "git", "status", "--porcelain", "--untracked-files=no"
], cwd=trainer_dir, text=True).strip():
    raise RuntimeError("Existing trainer checkout differs from the release; use a fresh working session.")
constraints = Path("/kaggle/working/lora-constraints.txt")
installed_torch = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        installed_torch.append(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        if package == "torch":
            raise RuntimeError("Use a Kaggle GPU image with PyTorch installed.")
constraints.write_text("\n".join(installed_torch) + "\n", encoding="utf-8")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--constraint", str(constraints),
    "-r", "requirements.txt", "numpy==1.26.4", "peft==0.17.1"
], cwd=trainer_dir, check=True)
print("Installed trainer commit:", head)
print("Restart the kernel now; continue at Step 2 without rerunning Step 1.")

## Step 2 — Configure the experiment

Edit `RUN` and `DATASET_PATH` together: RUN is a label, not a dataset selector.
Point DATASET_PATH at the exact image collection, either flat or nested. All images
below that path are included, so do not select a parent containing multiple experiments.
`CAPTION_MODE="uniform"` retains the original experiment; `"preserve"` requires an
existing, nonempty `.txt` sidecar for every image. Only staged copies are captioned.
Set EXPECTED_IMAGES to None for another dataset size.

To switch experiments, rerun this cell and every subsequent cell. Every preparation
creates a new output directory, preventing accidental checkpoint overwrites.

In [ ]:
import os
import re
import sys
import json
import subprocess
from pathlib import Path

RUN = "glaze"
DATASET_PATH = Path("/kaggle/input/datasets/leothiii/glaze-photos")
LOCAL_MODEL_PATH = None  # Optional path to an attached SDXL base .safetensors file.
EXPECTED_IMAGES = 30
CAPTION_MODE = "uniform"  # "uniform" or "preserve"
UNIFORM_CAPTION = "ohwx person"
REPEATS = 10
RESOLUTION = 512
RANK, ALPHA = 16, 8
BATCH_SIZE, GRAD_ACCUM = 1, 4
MAX_STEPS, WARMUP_STEPS = 500, 50
LEARNING_RATE = 1e-4
SEED = 42
SAVE_EVERY = 100
TRAINER_DIR = Path("/kaggle/working/sd-scripts-v0.10.6")
TRAIN_SCRIPT = TRAINER_DIR / "sdxl_train_network.py"

if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN):
    raise ValueError("RUN must contain only letters, digits, underscores or hyphens.")
if CAPTION_MODE not in {"uniform", "preserve"}:
    raise ValueError("Unknown CAPTION_MODE.")
if not DATASET_PATH.is_dir():
    raise FileNotFoundError(f"Set DATASET_PATH to your attached dataset: {DATASET_PATH}")
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError("Complete Step 1 first.")
if min(REPEATS, RESOLUTION, RANK, BATCH_SIZE, GRAD_ACCUM, MAX_STEPS, SAVE_EVERY) <= 0:
    raise ValueError("Training counts and dimensions must be positive.")
if RESOLUTION % 32 or not 0 <= WARMUP_STEPS < MAX_STEPS:
    raise ValueError("Resolution must be divisible by 32; warmup must be below MAX_STEPS.")
# Set visibility before any CUDA import and pass it to the training subprocess too.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
print(f"Run: {RUN}; dataset: {DATASET_PATH}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}; optimizer steps: {MAX_STEPS}")

## Step 3 — Verify the runtime and obtain the SDXL base model

The trainer's `--help` is checked in a fresh process to catch import and option errors
before downloading the checkpoint. The Hub cache path is used directly, avoiding a
second multi-GB copy. The resolved model revision is recorded for downloaded models.

In [ ]:
import torch
import importlib.metadata as metadata

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU and restart the kernel.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
runtime_env = os.environ.copy()
probe = subprocess.run([
    sys.executable, str(TRAIN_SCRIPT), "--help"
], cwd=TRAINER_DIR, env=runtime_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if probe.returncode:
    print(probe.stdout)
    raise RuntimeError("Trainer import failed. Check Step 1 output and restart the kernel.")
required_flags = ("--network_train_unet_only", "--cache_text_encoder_outputs", "--sdpa")
if any(flag not in probe.stdout for flag in required_flags):
    raise RuntimeError("Trainer does not support the required options.")
PACKAGE_VERSIONS = {name: metadata.version(name) for name in (
    "torch", "torchvision", "accelerate", "transformers", "diffusers",
    "huggingface-hub", "bitsandbytes", "numpy", "peft", "safetensors"
)}
print(json.dumps(PACKAGE_VERSIONS, indent=2))
MODEL_REVISION = None
if LOCAL_MODEL_PATH is not None:
    MODEL_PATH = Path(LOCAL_MODEL_PATH)
else:
    from huggingface_hub import HfApi, hf_hub_download
    repo_id = "stabilityai/stable-diffusion-xl-base-1.0"
    MODEL_REVISION = HfApi().model_info(repo_id, revision="main").sha
    MODEL_PATH = Path(hf_hub_download(
        repo_id=repo_id, revision=MODEL_REVISION,
        filename="sd_xl_base_1.0.safetensors",
        cache_dir="/kaggle/working/hf-cache"
    ))
if not MODEL_PATH.is_file() or MODEL_PATH.stat().st_size == 0:
    raise FileNotFoundError(f"Missing or empty model checkpoint: {MODEL_PATH}")
print("Base model:", MODEL_PATH)

## Step 4 — Validate and stage the dataset

Check actual image readability and captions, rather than comparing two file counts.
Staged filenames are unique even when source folders reuse names. Image bytes are
copied unchanged; training retains the original fixed square resize/crop behavior.
There is no random crop, flip, caption shuffle, or color augmentation, because cached
latents/text embeddings must remain consistent. Fresh staging avoids stale caches.

In [ ]:
import hashlib
import shutil
import tempfile
from PIL import Image
import toml

extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
images = sorted(p for p in DATASET_PATH.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
if not images:
    raise ValueError("No supported images found under DATASET_PATH.")
if EXPECTED_IMAGES is not None and len(images) != EXPECTED_IMAGES:
    raise ValueError(f"Expected {EXPECTED_IMAGES} images, found {len(images)}.")
if CAPTION_MODE == "uniform" and not UNIFORM_CAPTION.strip():
    raise ValueError("UNIFORM_CAPTION cannot be empty.")
records = []
for source in images:
    with Image.open(source) as image:
        image.verify()
    with Image.open(source) as image:
        image.load()
        width, height = image.size
    caption = UNIFORM_CAPTION.strip()
    if CAPTION_MODE == "preserve":
        sidecar = source.with_suffix(".txt")
        if not sidecar.is_file():
            raise FileNotFoundError(f"Missing caption: {sidecar}")
        caption = sidecar.read_text(encoding="utf-8-sig").strip()
        if not caption:
            raise ValueError(f"Empty caption: {sidecar}")
    records.append({
        "source": str(source.relative_to(DATASET_PATH)), "caption": caption,
        "width": width, "height": height,
        "sha256": hashlib.sha256(source.read_bytes()).hexdigest()
    })

output_root = Path("/kaggle/working/outputs")
output_root.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix=f"lora_run_{RUN}_", dir=output_root))
OUTPUT_NAME = f"lora_run_{RUN}"
TRAIN_DATA_DIR = OUTPUT_DIR / "training_data"
TRAIN_DATA_DIR.mkdir()
for index, (source, record) in enumerate(zip(images, records)):
    target = TRAIN_DATA_DIR / f"{index:05d}{source.suffix.lower()}"
    shutil.copyfile(source, target)
    target.with_suffix(".txt").write_text(record["caption"], encoding="utf-8")
    record["staged"] = target.name

DATASET_CONFIG = OUTPUT_DIR / "dataset.toml"
DATASET_CONFIG.write_text(toml.dumps({
    "general": {"caption_extension": ".txt", "shuffle_caption": False},
    "datasets": [{
        "resolution": RESOLUTION, "batch_size": BATCH_SIZE, "enable_bucket": False,
        "subsets": [{"image_dir": str(TRAIN_DATA_DIR), "num_repeats": REPEATS}]
    }]
}), encoding="utf-8")
(OUTPUT_DIR / "dataset_manifest.json").write_text(
    json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Validated {len(records)} images and captions. Output: {OUTPUT_DIR}")
print("Sizes:", sorted({(r["width"], r["height"]) for r in records}))

## Step 5 — Train and verify the saved adapter

Explicit U-Net-only LoRA avoids creating text-encoder adapters with zero learning rate.
Cached text embeddings let the trainer move the frozen text encoders off the GPU.
FP16, FP32 VAE, gradient checkpointing, SDPA, latent caching, and AdamW8bit are retained.
The script is launched directly using the kernel's Python in a single process, avoiding
a saved Accelerate launcher configuration that could select distributed execution.
The trainer still uses Accelerate internally for precision and gradient accumulation.

Logs, exact command, package versions, trainer commit, model revision and dataset hashes
are saved with the adapter. Periodic adapter files are inference checkpoints, **not full
optimizer-state resumes**. Download the output directory before ending the Kaggle session.

In [ ]:
import shlex
from datetime import datetime, timezone

final_adapter = OUTPUT_DIR / f"{OUTPUT_NAME}.safetensors"
if (OUTPUT_DIR / "run.json").exists():
    raise FileExistsError("This run was already launched. Rerun Step 4 for a fresh output directory.")
command = [
    sys.executable, "-u", str(TRAIN_SCRIPT),
    f"--pretrained_model_name_or_path={MODEL_PATH}",
    f"--dataset_config={DATASET_CONFIG}",
    f"--output_dir={OUTPUT_DIR}", f"--output_name={OUTPUT_NAME}",
    "--save_model_as=safetensors", "--network_module=networks.lora",
    f"--network_dim={RANK}", f"--network_alpha={ALPHA}",
    "--network_train_unet_only", "--cache_text_encoder_outputs",
    f"--gradient_accumulation_steps={GRAD_ACCUM}", f"--max_train_steps={MAX_STEPS}",
    f"--learning_rate={LEARNING_RATE}", f"--unet_lr={LEARNING_RATE}",
    "--lr_scheduler=constant_with_warmup", f"--lr_warmup_steps={WARMUP_STEPS}",
    "--optimizer_type=AdamW8bit", "--mixed_precision=fp16", "--save_precision=fp16",
    "--no_half_vae", "--gradient_checkpointing", "--sdpa", "--cache_latents",
    "--vae_batch_size=1", "--max_data_loader_n_workers=0",
    f"--save_every_n_steps={SAVE_EVERY}", f"--seed={SEED}",
    "--log_with=tensorboard", f"--logging_dir={OUTPUT_DIR / 'logs'}"
]
trainer_commit = subprocess.check_output([
    "git", "rev-parse", "HEAD"
], cwd=TRAINER_DIR, text=True).strip()
manifest = {
    "run": RUN, "dataset": str(DATASET_PATH), "caption_mode": CAPTION_MODE,
    "command": command, "packages": PACKAGE_VERSIONS, "trainer_commit": trainer_commit,
    "model_path": str(MODEL_PATH), "model_revision": MODEL_REVISION,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM, "seed": SEED,
    "started_utc": datetime.now(timezone.utc).isoformat(), "status": "running"
}
run_file = OUTPUT_DIR / "run.json"
run_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(OUTPUT_DIR / "command.txt").write_text(shlex.join(command), encoding="utf-8")
print(shlex.join(command), flush=True)
train_env = os.environ.copy()
# Remove inherited distributed-process settings; this is a one-GPU experiment.
for key in list(train_env):
    if key.startswith("ACCELERATE_") or key in {
        "RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"
    }:
        train_env.pop(key)
train_env.update(CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
process = None
try:
    with (OUTPUT_DIR / "train.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=TRAINER_DIR, env=train_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    if not final_adapter.is_file():
        raise RuntimeError(f"Trainer exited without producing {final_adapter}")
    from safetensors import safe_open
    with safe_open(str(final_adapter), framework="pt", device="cpu") as adapter:
        keys = list(adapter.keys())
        if not keys or not any("lora_unet" in key for key in keys):
            raise RuntimeError("Saved adapter has no U-Net LoRA tensors.")
    manifest["status"] = "completed"
    print(f"Saved adapter: {final_adapter} ({final_adapter.stat().st_size / 2**20:.1f} MiB)")
except BaseException as error:
    manifest["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    raise
finally:
    manifest["finished_utc"] = datetime.now(timezone.utc).isoformat()
    manifest["returncode"] = process.returncode if process is not None else None
    run_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

## Review notes and validation limits

Corrected: hardcoded training paths overriding RUN; conflicting dependency installs;
fallback from an unrelated version tag to main; incorrect installation working directory;
shell launches using a different Python; count-only dataset checks; missing captions;
deletion/reuse of previous staging; silent subprocess failures; and success without an
actual adapter. The notebook no longer claims that its settings were checked against an
unavailable paper table.

Memory improvements follow the trainer's supported caching and U-Net-only options.
No speedup, peak VRAM figure, model quality improvement, or successful Kaggle GPU run
is claimed without measurement. A fixed seed does not guarantee bitwise-identical CUDA
results. Compare every experiment with the same environment, captions, preprocessing,
step budget, base model and generation/evaluation settings.

References:
- [Pinned trainer requirements](https://github.com/kohya-ss/sd-scripts/blob/v0.10.6/requirements.txt)
- [SDXL training documentation](https://github.com/kohya-ss/sd-scripts/blob/v0.10.6/docs/sdxl_train_network.md)
- [SDXL trainer implementation](https://github.com/kohya-ss/sd-scripts/blob/v0.10.6/sdxl_train_network.py)